[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/03_leslie3d_example2.ipynb)

In [ ]:
# On Colab, install the CMGDB fork (a prebuilt wheel; not on PyPI)
# and the paper package. Running locally uses the project venv as is.
import sys

if "google.colab" in sys.modules:
    !pip install -q cmgdb==1.3.3+fork.2 --find-links https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3+fork.2
    !pip install -q git+https://github.com/begelb/latent_dynamics.git@paper

# Section 5.2.2 - Three-dimensional Leslie: bistability identified

## What this notebook shows

The same three-generation **Leslie model** (paper section 5.2) at parameters
where the method *succeeds*: the two-dimensional latent Morse graph has **two
minimal nodes** whose Conley indices $(x^4-1, 0, 0)$ are those of period-four
orbits, faithfully matching the true system's bistability.

Read this alongside notebook 02 (the spurious-attractor failure case) to see
both sides of the method.

### How to run

Edit the **parameters** cell below, then *Run All*. Three modes, run in
stages: model, training curves, Morse graph.

| `MODE` | what it does | cost |
|--------|--------------|------|
| `"replay"` | re-render the paper's saved Morse graph and Morse sets | seconds |
| `"morse"` | recompute the Morse graph of the *saved* model at your `SUBDIV` | seconds-minutes |
| `"retrain"` | train a fresh model with your `OVERRIDES`, then compute its Morse graph at `SUBDIV` | minutes-hours |

**Toy subdivisions are a qualitative preview.** Coarse CMGDB grids can merge
nearby recurrent sets and change the Morse graph; the paper figures use the
config's (finer) values. The paper value for this example is noted in the
parameters cell.

In [ ]:
# ===== PARAMETERS  (edit, then Run All) ====================================
MODE = "replay"            # "replay" | "morse" | "retrain"
SEED = None                # None -> the config's default seed
SUBDIV = (10, 14, 20)      # MODE="morse"/"retrain": (subdiv_init, subdiv_min, subdiv_max)
                           # paper value: (25, 28, 29)
BOX_SCALE = "auto"         # Morse-set box size: "auto" | float | {label: float}
# ===========================================================================

REPLAY_CONFIG  = "leslie3d_example2_replay"
RETRAIN_CONFIG = "leslie3d_example2"


## The system

The same three-generation Leslie map as the previous notebook,

$$f(x) = \big((\theta_1 x_1 + \theta_2 x_2 + \theta_3 x_3)\,e^{-0.1(x_1+x_2+x_3)},\;
p_1 x_1,\; p_2 x_2\big)$$

at the same parameters. What differs is the model: a wider network on a
different data design, whose latent Morse graph recovers the true bistability
instead of an extra attractor.

In [ ]:
# ---- the system (paper values; edit and re-run to explore) ----------------
TH1, TH2, TH3 = 28.9, 29.8, 22.0    # fecundities
SURVIVAL_1, SURVIVAL_2 = 0.7, 0.7   # survival between generations
LATENT_DIMS = 2

from latentdynamics.config import load_config
from latentdynamics.systems import build_system

SYSTEM_PARAMS = {
    "th1": TH1, "th2": TH2, "th3": TH3,
    "survival_p1": SURVIVAL_1, "survival_p2": SURVIVAL_2,
}
system = build_system("leslie3d", SYSTEM_PARAMS)
print(f"ambient dimension {system.dim}, latent dimension {LATENT_DIMS}")
print(f"phase space  lower {system.lower_bounds.tolist()}")
print(f"             upper {system.upper_bounds.tolist()}")

## The autoencoder and its latent map

An encoder, a decoder, and a latent map trained together so the latent map is
an $\epsilon$-approximate semiconjugacy to the full system on the data.

In [ ]:
# ---- the autoencoder (paper values) --------------------------------------
HIDDEN_SHAPES = [64, 64]            # per component: encoder, latent map, decoder
LOSS_WEIGHTS = [100.0, 10.0, 20.0]  # (w1, w2, w3): reconstruction, latent step, cycle
LEARNING_RATE = 1e-3
BATCH_SIZE = 1024
EPOCHS = 1000
PATIENCE = 100

print(f"{system.dim} -> {LATENT_DIMS} -> {system.dim}, hidden {HIDDEN_SHAPES} per component")
print(f"loss weights {LOSS_WEIGHTS}, Adam lr {LEARNING_RATE}, batch {BATCH_SIZE}")

## The data

Pairs $(x, f(x))$ along trajectories from sampled initial conditions: the model
only ever sees one-step transitions, never the map itself.

In [ ]:
# ---- the data (paper values) ---------------------------------------------
N_TRAIN = 8000        # training trajectories
N_VAL = 2000          # validation trajectories
N_ITERATIONS = 20     # steps per trajectory

print(f"{N_TRAIN} train / {N_VAL} validation trajectories, {N_ITERATIONS} steps each")

# Everything above is fed to the pipeline as config overrides, so `retrain`
# below trains exactly the model described here.
OVERRIDES = {
    "system": {"params": SYSTEM_PARAMS},
    "arch": {
        "low_dims": LATENT_DIMS,
        "encoder": {"hidden_shapes": HIDDEN_SHAPES},
        "latent_map": {"hidden_shapes": HIDDEN_SHAPES},
        "decoder": {"hidden_shapes": HIDDEN_SHAPES},
    },
    "training": {
        "loss_weights": LOSS_WEIGHTS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "patience": PATIENCE,
    },
    "data": {
        "n_samples_val": N_VAL,
        "n_iterations": N_ITERATIONS,
        
    },
}

# Flag anything that no longer matches the paper's configuration.
paper = load_config(RETRAIN_CONFIG)
drift = []
if SYSTEM_PARAMS != paper.system.params:
    drift.append(f"system {paper.system.params}")
if LATENT_DIMS != paper.arch.low_dims:
    drift.append(f"latent dims {paper.arch.low_dims}")
if HIDDEN_SHAPES != paper.arch.encoder.hidden_shapes:
    drift.append(f"encoder hidden {paper.arch.encoder.hidden_shapes}")
for name, value, reference in [
    ("loss weights", LOSS_WEIGHTS, paper.training.loss_weights),
    ("learning rate", LEARNING_RATE, paper.training.learning_rate),
    ("batch size", BATCH_SIZE, paper.training.batch_size),
    ("epochs", EPOCHS, paper.training.epochs),
    ("patience", PATIENCE, paper.training.patience),
    ("validation trajectories", N_VAL, paper.data.n_samples_val),
    ("iterations", N_ITERATIONS, paper.data.n_iterations),
]:
    if value != reference:
        drift.append(f"{name} {reference}")
print("\nmatches the paper's configuration" if not drift
      else "\ndiffers from the paper, which uses: " + "; ".join(drift))

## Training

`replay` and `morse` load the paper's trained weights. `retrain` runs the data,
scaling, training and diagnostic stages -- and nothing else, so CMGDB below is
free to use a grid this machine can afford.

In [ ]:
from latentdynamics.replay import load_experiment, retrain

if MODE in ("replay", "morse"):
    exp = load_experiment(REPLAY_CONFIG, seed=SEED)
elif MODE == "retrain":
    # Training only. CMGDB runs further down, so a long training run survives a
    # Morse computation that needs a different grid.
    exp = retrain(
        RETRAIN_CONFIG,
        seed=SEED,
        overrides=OVERRIDES,
        stages=("data", "scale", "train", "diagnose"),
    )
else:
    raise ValueError(f"unknown MODE {MODE!r}")
exp


## Training curves

Total loss and its terms, per epoch. In `replay` and `morse` mode these are the
saved curves of the run the paper reports; in `retrain` mode they are the run
that just finished. Training stops early when validation loss stalls, so the
curves usually end well before the configured epoch budget.

In [ ]:
import json

import matplotlib.pyplot as plt

history_path = exp.seed_dir / "logs" / "history.json"
if not history_path.exists():
    print(f"no training history at {history_path}")
else:
    history = json.loads(history_path.read_text())
    terms = [k for k in history["train"] if k != "loss_total"]

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    left.semilogy(history["train"]["loss_total"], label="train")
    left.semilogy(history["val"]["loss_total"], label="validation")
    left.set(xlabel="epoch", ylabel="total loss")
    left.legend()

    for term in terms:
        right.semilogy(history["val"][term], label=term)
    right.set(xlabel="epoch", ylabel="validation loss by term")
    right.legend(fontsize="small")

    fig.tight_layout()
    plt.show()
    print(f"{len(history['train']['loss_total'])} epochs")

## Morse graph

CMGDB subdivides the latent rectangle into boxes and builds the directed graph
on them induced by the latent map: box `B` points at every box meeting an
enclosure of `g(B)`. The **Morse sets** are that graph's strongly connected
components, and the **Morse graph** is its condensation, with a Conley index on
each node. `ComputeConleyMorseGraph` returns both -- the map graph is not an
extra step, it *is* the computation.

`replay` reads the paper's saved artifacts. Otherwise the grid comes from
`SUBDIV`, chosen here rather than inherited from the training config.

The box map evaluates the network on the corner lattice in batches -- one call
per batch instead of one per box. Installing that batched map also caches the
graph's adjacencies in a single block, bounded by `CMGDB_MAPGRAPH_MAX_VERTICES`
(`2**24` cells) and `CMGDB_MAPGRAPH_MAX_EDGES` (`2*10**8` edges, 8 bytes each).
Both are read from the environment on every run, so a large-memory machine can
raise them:

```python
import os
os.environ["CMGDB_MAPGRAPH_MAX_EDGES"] = "1200000000"      # 9.6 GB of edges
os.environ["CMGDB_MAPGRAPH_RESERVE_EDGES"] = "1200000000"  # allocate once
```

Exceeding a limit stops the run rather than falling back to a per-box callback,
which would be orders of magnitude slower.

In [ ]:
import CMGDB
import numpy as np

from latentdynamics.replay import repo_path, show_image
from latentdynamics.viz import render_morse_from_files, save_morse_graph_artifacts

cmgdb_cfg = exp.seed_cfg.cmgdb

if MODE == "replay":
    MG_DIR = exp.morse_dir
    lower_bounds, upper_bounds = exp.morse_bounds()
    print(f"replay: the paper's saved Morse graph in {MG_DIR}")
else:
    subdiv_init, subdiv_min, subdiv_max = SUBDIV

    # Latent bounding box: encode current and next states, pad the range.
    high = exp.arch.high_dims
    train = np.loadtxt(exp.data_csv, delimiter=",", skiprows=1, ndmin=2)
    val = np.loadtxt(exp.seed_cfg.paths.val_csv(), delimiter=",", skiprows=1, ndmin=2)
    z = np.vstack([
        exp.encode(block)
        for block in (train[:, :high], train[:, high:], val[:, :high], val[:, high:])
    ])
    z_min, z_max = z.min(axis=0), z.max(axis=0)
    delta = cmgdb_cfg.bounds_epsilon_frac * (z_max - z_min)
    lower_bounds = (z_min - delta).tolist()
    upper_bounds = (z_max + delta).tolist()

    # F(rect) encloses g(rect), evaluated on the corner lattice in batches.
    box_map = CMGDB.make_precomputed_box_map(
        exp.model.latent_map,
        lower_bounds,
        upper_bounds,
        subdiv_max=subdiv_max,
        mode="adaptive",
        padding=cmgdb_cfg.padding,
        max_table_points=cmgdb_cfg.max_table_points,
        device="auto",
    )

    dyn_model = CMGDB.Model(
        subdiv_min,
        subdiv_max,
        subdiv_init,
        cmgdb_cfg.subdiv_limit,
        lower_bounds,
        upper_bounds,
        box_map,
    )
    # Installing the batched map is what caches the adjacencies in one block.
    dyn_model.set_batch_map(box_map.batch)

    morse_graph, map_graph = CMGDB.ComputeConleyMorseGraph(dyn_model)

    MG_DIR = repo_path(
        "output", "notebooks", exp.name,
        f"morse_{subdiv_init}-{subdiv_min}-{subdiv_max}", "MG",
    )
    save_morse_graph_artifacts(morse_graph, MG_DIR)
    print(f"bounds {lower_bounds} -> {upper_bounds}")
    print(f"artifacts -> {MG_DIR}")

### What came out

Each Morse set with its Conley index, how many boxes it occupies, and what it
flows into. A node with no outgoing edges is minimal: an attractor.

In [ ]:
from latentdynamics.analysis import MorseGraph

graph = MorseGraph.from_dot(MG_DIR / "morse_graph")
boxes = np.atleast_2d(np.loadtxt(MG_DIR / "morse_sets", delimiter=","))
counts = dict(zip(*np.unique(boxes[:, -1].astype(int), return_counts=True)))

print(f"{len(graph.nodes)} Morse sets, {len(graph.minimal)} minimal")
for node in graph.nodes:
    index = graph.labels.get(node, "?").split(":", 1)[-1].strip()
    edges = sorted(graph.edges.get(node, []))
    flow = "minimal" if not edges else "-> " + ", ".join(str(e) for e in edges)
    print(f"  {node}: {index:<16} {counts.get(node, 0):>8d} boxes   {flow}")

## Figures

Rendered from the DOT and CSV above with the paper's palette and axis labels,
so a recomputed run and the paper figure come out of the same code. `BOX_SCALE`
only affects drawing: it inflates Morse sets too small to see.

In [ ]:
figs = render_morse_from_files(
    MG_DIR,
    bounds_lower=lower_bounds,
    bounds_upper=upper_bounds,
    out_dir=repo_path("notebooks", "rendered", exp.name),
    box_scale=BOX_SCALE,
)
show_image(figs.morse_graph_png, width=600)
for png in (p for p in figs.morse_sets_paths if p.suffix == ".png"):
    show_image(png, width=720)

## Run provenance

In [ ]:
exp.diagnostics()